In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install timm scikit-learn tqdm --quiet

In [ ]:
import os
import pandas as pd
import numpy as np

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn

from torchvision import transforms

import timm

In [ ]:
torch.set_num_threads(2)
torch.set_grad_enabled(False)

device = torch.device("cpu")

print("Device:", device)

Device: cpu


In [ ]:
save_dir = "/content/drive/MyDrive/weighted_fusion_results"
os.makedirs(save_dir, exist_ok=True)

print("Saving results to:", save_dir)

Saving results to: /content/drive/MyDrive/weighted_fusion_results


In [ ]:
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std  = [0.229, 0.224, 0.225]

dr_transform = transforms.Compose([
    transforms.Resize((300,300)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

common_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

In [ ]:
import timm
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

dr_model = timm.create_model(
    "tf_efficientnetv2_s",
    pretrained=False,
    num_classes=1
)

ckpt = torch.load(
    "/content/drive/MyDrive/dr_results/final_dr_model_best.pth",
    map_location=device
)

state_dict = ckpt["model"] if "model" in ckpt else ckpt

new_state_dict = {}
for k, v in state_dict.items():
    new_state_dict[k.replace("module.", "")] = v

dr_model.load_state_dict(new_state_dict)

dr_model.to(device)
dr_model.eval()

print("✅ DR model loaded")

✅ DR model loaded


In [ ]:
gl_model = timm.create_model("convnext_tiny", pretrained=False, num_classes=2)

gl_path = "/content/drive/MyDrive/Glaucoma_results/best_model.pth"

gl_model.load_state_dict(torch.load(gl_path, map_location=device))
gl_model.eval().to(device)

ConvNeXt(
  (stem): Sequential(
    (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
    (1): LayerNorm2d((96,), eps=1e-06, elementwise_affine=True)
  )
  (stages): Sequential(
    (0): ConvNeXtStage(
      (downsample): Identity()
      (blocks): Sequential(
        (0): ConvNeXtBlock(
          (conv_dw): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (norm): LayerNorm((96,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
            (fc1): Linear(in_features=96, out_features=384, bias=True)
            (act): GELU()
            (drop1): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (fc2): Linear(in_features=384, out_features=96, bias=True)
            (drop2): Dropout(p=0.0, inplace=False)
          )
          (shortcut): Identity()
          (drop_path): Identity()
        )
        (1): ConvNeXtBlock(
          (conv_dw): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)


In [ ]:
amd_model = timm.create_model("densenet121", pretrained=False, num_classes=1)

amd_path = "/content/drive/MyDrive/amd_results/best_model_combined.pth"

amd_model.load_state_dict(torch.load(amd_path, map_location=device))
amd_model.eval().to(device)

DenseNet(
  (features): Sequential(
    (conv0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (norm0): BatchNormAct2d(
      64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
      (drop): Identity()
      (act): ReLU(inplace=True)
    )
    (pool0): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (denseblock1): DenseBlock(
      (denselayer1): DenseLayer(
        (norm1): BatchNormAct2d(
          64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): ReLU(inplace=True)
        )
        (conv1): Conv2d(64, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (norm2): BatchNormAct2d(
          128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): ReLU(inplace=True)
        )
        (conv2): Conv2d(128, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      )
  

In [ ]:
ded_model = timm.create_model("efficientnet_b0", pretrained=False, num_classes=1)

ded_path = "/content/drive/MyDrive/ded_results/best_ded_model.pth"

ded_model.load_state_dict(torch.load(ded_path, map_location=device))
ded_model.eval().to(device)

EfficientNet(
  (conv_stem): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
  (bn1): BatchNormAct2d(
    32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
    (drop): Identity()
    (act): SiLU(inplace=True)
  )
  (blocks): Sequential(
    (0): Sequential(
      (0): DepthwiseSeparableConv(
        (conv_dw): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (bn1): BatchNormAct2d(
          32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True
          (drop): Identity()
          (act): SiLU(inplace=True)
        )
        (aa): Identity()
        (se): SqueezeExcite(
          (conv_reduce): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
          (act1): SiLU(inplace=True)
          (conv_expand): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
          (gate): Sigmoid()
        )
        (conv_pw): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn2

In [ ]:
csv_path = "/content/drive/MyDrive/multimodal_unified/train.csv"

df = pd.read_csv(csv_path)

print("Total rows:", len(df))
df.head()

Total rows: 20970


,image_path,modality,DR,Glaucoma,AMD,DED
0,E:\Datasets\fundus_unified\images/GLAU_train_1...,fundus,0,1,0,0
1,E:\Datasets\fundus_unified\images/GLAU_train_0...,fundus,0,0,0,0
2,E:\Datasets\fundus_unified\images/GLAU_train_0...,fundus,0,0,0,0
3,E:\Datasets\fundus_unified\images/DR_aptos_b7c...,fundus,0,0,0,0
4,E:\Datasets\fundus_unified\images/GLAU_val_0_O...,fundus,0,0,0,0


In [ ]:
fundus_root = "/content/drive/MyDrive/fundus_unified/images"
oct_root = "/content/drive/MyDrive/oct_unified/images"
slit_root = "/content/drive/MyDrive/ded_unified/images"

In [ ]:
def rebuild_path(row):

    filename = os.path.basename(row["image_path"])

    if row["modality"] == "fundus":
        return os.path.join(fundus_root, filename)

    elif row["modality"] == "oct":
        return os.path.join(oct_root, filename)

    elif row["modality"] == "slitlamp":
        return os.path.join(slit_root, filename)

    return None

In [ ]:
def load_images(paths, transform):

    imgs = []

    for p in paths:
        img = Image.open(p).convert("RGB")
        img = transform(img)
        imgs.append(img)

    return torch.stack(imgs)

In [ ]:
from tqdm import tqdm
import torch
from PIL import Image
import pandas as pd
import os

batch_size = 16

records = []

save_dir = "/content/drive/MyDrive/weighted_fusion_results"
checkpoint_path = os.path.join(save_dir, "fusion_train_checkpoint.csv")

for start in tqdm(range(0, len(df), batch_size)):

    batch = df.iloc[start:start+batch_size]

    fundus_paths, fundus_idx = [], []
    oct_paths, oct_idx = [], []
    slit_paths, slit_idx = [], []

    for i,row in batch.iterrows():

        p = rebuild_path(row)

        if not os.path.exists(p):
            continue

        if row.modality == "fundus":
            fundus_paths.append(p)
            fundus_idx.append(i)

        elif row.modality == "oct":
            oct_paths.append(p)
            oct_idx.append(i)

        elif row.modality == "slitlamp":
            slit_paths.append(p)
            slit_idx.append(i)

    dr_probs, gl_probs = {}, {}
    amd_probs, ded_probs = {}, {}

    with torch.no_grad():

        # ---------- FUNDUS ----------
        if fundus_paths:

            imgs_raw = [Image.open(p).convert("RGB") for p in fundus_paths]

            imgs_dr = torch.stack([dr_transform(img) for img in imgs_raw]).to(device)
            imgs_gl = torch.stack([common_transform(img) for img in imgs_raw]).to(device)

            # 🔥 NEW DR MODEL
            dr_out = torch.sigmoid(dr_model(imgs_dr))

            for k,i in enumerate(fundus_idx):
                dr_probs[i] = dr_out[k].item()

            # Glaucoma
            gl_out = torch.softmax(gl_model(imgs_gl),1)

            for k,i in enumerate(fundus_idx):
                gl_probs[i] = gl_out[k,1].item()

        # ---------- OCT ----------
        if oct_paths:

            imgs_raw = [Image.open(p).convert("RGB") for p in oct_paths]
            imgs = torch.stack([common_transform(img) for img in imgs_raw]).to(device)

            amd_out = torch.sigmoid(amd_model(imgs))

            for k,i in enumerate(oct_idx):
                amd_probs[i] = amd_out[k].item()

        # ---------- SLITLAMP ----------
        if slit_paths:

            imgs_raw = [Image.open(p).convert("RGB") for p in slit_paths]
            imgs = torch.stack([common_transform(img) for img in imgs_raw]).to(device)

            ded_out = torch.sigmoid(ded_model(imgs))

            for k,i in enumerate(slit_idx):
                ded_probs[i] = ded_out[k].item()

    # ---------- SAVE RECORDS ----------
    for i,row in batch.iterrows():

        rec = {
            "DR_prob": dr_probs.get(i,0),
            "Gl_prob": gl_probs.get(i,0),
            "AMD_prob": amd_probs.get(i,0),
            "DED_prob": ded_probs.get(i,0),

            "is_fundus": int(row.modality=="fundus"),
            "is_oct": int(row.modality=="oct"),
            "is_slitlamp": int(row.modality=="slitlamp"),

            "DR": row.DR,
            "Glaucoma": row.Glaucoma,
            "AMD": row.AMD,
            "DED": row.DED
        }

        records.append(rec)

    # 🔥 CHECKPOINT SAVE EVERY 100 BATCHES
    if start % (batch_size * 100) == 0 and start != 0:

        pd.DataFrame(records).to_csv(checkpoint_path, index=False)

        print(f"Checkpoint saved at {start} samples")

  8%|▊         | 101/1311 [12:51<2:12:04,  6.55s/it]

Checkpoint saved at 1600 samples


 15%|█▌        | 201/1311 [24:31<2:10:51,  7.07s/it]

Checkpoint saved at 3200 samples


 23%|██▎       | 301/1311 [36:29<1:56:16,  6.91s/it]

...

 92%|█████████▏| 1201/1311 [2:20:24<13:06,  7.15s/it]

Checkpoint saved at 19200 samples


 99%|█████████▉| 1301/1311 [2:32:03<01:16,  7.61s/it]

Checkpoint saved at 20800 samples


100%|██████████| 1311/1311 [2:33:13<00:00,  7.01s/it]


In [ ]:
import pandas as pd
import os

save_dir = "/content/drive/MyDrive/weighted_fusion_results"
os.makedirs(save_dir, exist_ok=True)

final_train_path = os.path.join(save_dir,"fusion_train.csv")

pd.DataFrame(records).to_csv(final_train_path,index=False)

print("Saved:", final_train_path)

Saved: /content/drive/MyDrive/weighted_fusion_results/fusion_train.csv


Testing the fusion_train!!

In [ ]:
import pandas as pd

train_features = pd.read_csv("/content/drive/MyDrive/weighted_fusion_results/fusion_train.csv")

print("Rows:", len(train_features))
train_features.head()

Rows: 20970


,DR_prob,Gl_prob,AMD_prob,DED_prob,is_fundus,is_oct,is_slitlamp,DR,Glaucoma,AMD,DED
0,0.015949,0.928803,0.0,0.0,1,0,0,0,1,0,0
1,0.002240,0.077405,0.0,0.0,1,0,0,0,0,0,0
2,0.014328,0.221050,0.0,0.0,1,0,0,0,0,0,0
3,0.000008,0.055887,0.0,0.0,1,0,0,0,0,0,0
4,0.115468,0.041344,0.0,0.0,1,0,0,0,0,0,0


In [ ]:
train_features.groupby("is_fundus")[["DR_prob","Gl_prob","AMD_prob","DED_prob"]].mean()

,DR_prob,Gl_prob,AMD_prob,DED_prob
is_fundus,,,,
0,0.000000,0.000000,0.523322,0.031772
1,0.363651,0.396151,0.000000,0.000000


In [ ]:
train_features[["DR","Glaucoma","AMD","DED"]].sum()

,0
DR,1300
Glaucoma,3875
AMD,4655
DED,264


In [ ]:
from sklearn.metrics import roc_auc_score

print("DR AUC:",
roc_auc_score(train_features["DR"],train_features["DR_prob"]))

print("Glaucoma AUC:",
roc_auc_score(train_features["Glaucoma"],train_features["Gl_prob"]))

print("AMD AUC:",
roc_auc_score(train_features["AMD"],train_features["AMD_prob"]))

print("DED AUC:",
roc_auc_score(train_features["DED"],train_features["DED_prob"]))

DR AUC: 0.9915465761996012
Glaucoma AUC: 0.9765693949372105
AMD AUC: 0.9996080653013822
DED AUC: 0.9999504242658401


In [ ]:
train_features.isna().sum()

,0
DR_prob,0
Gl_prob,0
AMD_prob,0
DED_prob,0
is_fundus,0
is_oct,0
is_slitlamp,0
DR,0
Glaucoma,0
AMD,0


In [ ]:
train_features[["DR_prob","Gl_prob","AMD_prob","DED_prob"]].describe()

,DR_prob,Gl_prob,AMD_prob,DED_prob
count,20970.000000,20970.000000,20970.000000,20970.000000
mean,0.217861,0.237332,0.209803,0.012738
std,0.363602,0.348874,0.398939,0.111171
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000
50%,0.001695,0.046915,0.000000,0.000000
75%,0.264243,0.328537,0.005407,0.000000
max,1.000000,0.992619,1.000000,1.000000


In [ ]:
train_features[["is_fundus","is_oct","is_slitlamp"]].sum()

,0
is_fundus,12563
is_oct,7982
is_slitlamp,425


In [ ]:
(train_features[["is_fundus","is_oct","is_slitlamp"]].sum(axis=1) != 1).sum()

np.int64(0)